# Selenium을 이용한 기상청 날씨 크롤링

In [1]:
%pip install -q selenium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.2 -> 25.0.1
[notice] To update, run: c:\Users\dandycode\.pyenv\pyenv-win\versions\3.11.7\python.exe -m pip install --upgrade pip


In [2]:
# 봇 처럼 여겨지지 않기 위해 주피터 노트북 ipynb 파일 생성
# 크롤링은 어떻게 사이트에서 사람이 하는 것처럼 보일까가 중요

# pip install selenium
from selenium import webdriver

driver = webdriver.Chrome()
# driver.set_window_size(1920, 1080)
driver.set_window_size(1280, 720)

# URL='https://www.naver.com/'
URL='https://www.weather.go.kr/w/weather/forecast/short-term.do'
driver.get(url=URL)

In [3]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


In [4]:
# --- 방법 1: 링크 텍스트 사용 (가장 간단하고 추천) ---
# "1시간 간격"이라는 텍스트를 가진 링크를 직접 찾습니다.
print("방법 1: 링크 텍스트로 클릭 시도...")
# WebDriverWait를 사용하여 요소가 클릭 가능할 때까지 최대 10초간 기다립니다.
one_hour_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.LINK_TEXT, "1시간 간격"))
)
one_hour_button.click()
print("'1시간 간격' 버튼 클릭 성공 (링크 텍스트 사용)")

방법 1: 링크 텍스트로 클릭 시도...
'1시간 간격' 버튼 클릭 성공 (링크 텍스트 사용)


In [6]:
# --- 방법 2: CSS 선택자 사용 ---
print("방법 2: CSS 선택자(클래스)로 클릭 시도...")
table_view_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "a.view-table"))
)
table_view_button.click()
print("'표 형태' 버튼 클릭 성공 (CSS 선택자 - 클래스 사용)")

방법 2: CSS 선택자(클래스)로 클릭 시도...
'표 형태' 버튼 클릭 성공 (CSS 선택자 - 클래스 사용)


In [7]:
from selenium.common.exceptions import NoSuchElementException
import re # 정규표현식 사용 (데이터 정제용)

In [8]:
# --- 데이터 저장을 위한 빈 리스트 초기화 ---
times = []
weathers = []
temperatures = []
feels_like_temps = []
precip_amounts = []
precip_intensities = []
precip_probabilities = []
wind_directions = []
wind_speeds = []
humidities = []
heatwave_impacts = []

# --- 데이터 추출 로직 ---
try:
    # 데이터 항목들을 포함하는 부모 div 찾기
    # daily_div = driver.find_element(By.CSS_SELECTOR, "div.daily")
    # item_wrap = daily_div.find_element(By.CSS_SELECTOR, "div.item-wrap")
    # 위 코드를 > 를 이용해 한줄로 작성 가능
    item_wrap = driver.find_element(By.CSS_SELECTOR, "div.daily > div.item-wrap")

    # print(item_wrap.get_attribute('outerHTML')) # item_wrap 내용 확인

    # 각 시간대별 데이터 묶음 (ul 태그) 찾기
    item_list = item_wrap.find_elements(By.CSS_SELECTOR, "ul.item")

    print(f"총 {len(item_list)}개의 시간대 데이터를 찾았습니다.")

    # 각 시간대별로 반복 처리
    for item_ul in item_list:
        # 각 ul 내의 li 요소들을 리스트로 가져오기
        # IndexError를 방지하기 위해 li 개수를 먼저 확인하는 것이 더 안전할 수 있습니다.
        try:
             li_elements = item_ul.find_elements(By.TAG_NAME, "li")
             # 최소 필요한 li 개수(예: 10개) 확인 로직 추가 가능
             # if len(li_elements) < 10: continue # 또는 None 추가 후 다음 item으로
        except NoSuchElementException:
             print("경고: 현재 시간대(ul.item)에서 li 요소들을 찾을 수 없습니다. 건너<0xEB><0x8D>니다.")
             # 모든 리스트에 None 추가하고 다음 item_ul로 넘어감
             lists_to_update = [times, weathers, temperatures, feels_like_temps, precip_amounts, precip_intensities, precip_probabilities, wind_directions, wind_speeds, humidities, heatwave_impacts]
             for lst in lists_to_update:
                 lst.append(None)
             continue # 다음 시간대로

        # 각 항목 추출 및 정제 (clean_value 함수 로직 인라인)

        # 1. 시각
        cleaned_time = None
        try:
            time_text = li_elements[0].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            # 정제 로직 (기본 전처리)
            time_text = time_text.strip().replace('&nbsp;', '')
            if time_text and time_text != '-':
                cleaned_time = time_text # 시각은 특별한 숫자 변환 없음
        except (NoSuchElementException, IndexError) as e:
             print(f"시각 처리 오류: {e}") # 디버깅용 로그
        times.append(cleaned_time)


        # 2. 날씨
        cleaned_weather = None
        try:
            # 먼저 wic 클래스 시도
            try:
                 weather_text = li_elements[1].find_element(By.CSS_SELECTOR, "span.wic").text
            except NoSuchElementException:
                 # wic 없으면 다른 span 시도
                 weather_text = li_elements[1].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            
            # 정제 로직 (기본 전처리)
            weather_text = weather_text.strip().replace('&nbsp;', '')
            if weather_text and weather_text != '-':
                 cleaned_weather = weather_text # 날씨는 텍스트 그대로
        except (NoSuchElementException, IndexError) as e:
            print(f"날씨 처리 오류: {e}")
        weathers.append(cleaned_weather)


        # 3. 기온 
        # 참고: 원래 코드에서는 li_elements[2] (3번째 li)에서 추출했으나, 
        # 이전 논의에서 실제 기온은 4번째 li에서 가져오는 것이 맞다고 판단했습니다.
        # 만약 3번째 li의 텍스트에서 첫 숫자를 기온으로 사용하려면 아래 로직 사용
        cleaned_temp = None
        try:
             # 3번째 li의 전체 텍스트 (예: "16℃(16℃)") 에서 첫 숫자 추출
             temp_text_combined = li_elements[2].find_element(By.CSS_SELECTOR, "span.hid.feel").text
             temp_text_combined = temp_text_combined.strip().replace('&nbsp;', '')
             if temp_text_combined and temp_text_combined != '-':
                 match = re.search(r'-?\d+', temp_text_combined) # 첫 번째 숫자 검색
                 if match:
                     cleaned_temp = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"기온 처리 오류: {e}")
        temperatures.append(cleaned_temp)
        

        # 4. 기온 체감 (3번째 li의 span.chill 텍스트)
        cleaned_feels_like = None
        try:
            chill_text = li_elements[2].find_element(By.CSS_SELECTOR, "span.chill").text # 예: (16℃)
            chill_text = chill_text.strip().replace('&nbsp;', '')
            if chill_text and chill_text != '-':
                match = re.search(r'-?\d+', chill_text) # 괄호 안 숫자 검색
                if match:
                    cleaned_feels_like = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"체감기온 처리 오류: {e}")
        feels_like_temps.append(cleaned_feels_like)


        # 5. 강수량
        cleaned_precip_amount = None
        try:
            pcp_text = li_elements[4].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            pcp_text = pcp_text.strip().replace('&nbsp;', '')
            if pcp_text and pcp_text != '-':
                if '빗방울' in pcp_text:
                    cleaned_precip_amount = 0.0 # '빗방울'은 0.0으로 처리
                else:
                    match = re.search(r'\d+\.?\d*', pcp_text) # 소수점 포함 숫자 검색
                    if match:
                        cleaned_precip_amount = float(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"강수량 처리 오류: {e}")
        precip_amounts.append(cleaned_precip_amount)


        # 6. 강수강도
        cleaned_intensity = None
        try:
            intensity_element = li_elements[5]
            intensity_text = None
            # 먼저 span 찾기 시도
            try:
                intensity_text = intensity_element.find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            except NoSuchElementException:
                 # span 없으면 li 전체 텍스트에서 hid 제외
                 full_text = intensity_element.text
                 hidden_text = ""
                 try:
                     hidden_text = intensity_element.find_element(By.CSS_SELECTOR, "span.hid").text
                 except NoSuchElementException: pass
                 intensity_text = full_text.replace(hidden_text, "").strip()

            # 정제 로직 (기본 전처리)
            intensity_text = intensity_text.strip().replace('&nbsp;', '')
            if intensity_text and intensity_text != '-':
                cleaned_intensity = intensity_text # 텍스트 그대로
        except (NoSuchElementException, IndexError) as e:
             print(f"강수강도 처리 오류: {e}")
        precip_intensities.append(cleaned_intensity)


        # 7. 강수확률
        cleaned_prob = None
        try:
            prob_text = li_elements[6].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            prob_text = prob_text.strip().replace('&nbsp;', '')
            if prob_text and prob_text != '-':
                match = re.search(r'\d+', prob_text) # % 제거 후 숫자만
                if match:
                    cleaned_prob = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"강수확률 처리 오류: {e}")
        precip_probabilities.append(cleaned_prob)


        # 8. 바람 (방향, 속도 분리)
        cleaned_wind_dir = None
        cleaned_wind_spd = None
        try:
            wind_li = li_elements[7]
            # 바람 방향
            try:
                wind_dir_text = wind_li.find_element(By.CSS_SELECTOR, "span.wdic").text
                wind_dir_text = wind_dir_text.strip().replace('&nbsp;', '')
                if wind_dir_text and wind_dir_text != '-':
                     cleaned_wind_dir = wind_dir_text
            except NoSuchElementException: pass # 없으면 None 유지
            # 바람 속도
            try:
                wind_spd_text = wind_li.find_element(By.CSS_SELECTOR, "span.wspd:not(.qwsd)").text
                wind_spd_text = wind_spd_text.strip().replace('&nbsp;', '')
                if wind_spd_text and wind_spd_text != '-':
                    match = re.search(r'\d+', wind_spd_text) # m/s 제거 후 숫자만
                    if match:
                        cleaned_wind_spd = int(match.group(0))
            except NoSuchElementException: pass # 없으면 None 유지
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"바람 처리 오류: {e}")
        wind_directions.append(cleaned_wind_dir)
        wind_speeds.append(cleaned_wind_spd)


        # 9. 습도
        cleaned_humidity = None
        try:
            hum_text = li_elements[8].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            hum_text = hum_text.strip().replace('&nbsp;', '')
            if hum_text and hum_text != '-':
                match = re.search(r'\d+', hum_text) # % 제거 후 숫자만
                if match:
                    cleaned_humidity = int(match.group(0))
        except (NoSuchElementException, IndexError, ValueError) as e:
            print(f"습도 처리 오류: {e}")
        humidities.append(cleaned_humidity)


        # 10. 폭염 영향
        cleaned_heatwave = None
        try:
            heat_text = li_elements[9].find_element(By.CSS_SELECTOR, "span:not(.hid)").text
            heat_text = heat_text.strip().replace('&nbsp;', '')
            if heat_text and heat_text != '-':
                cleaned_heatwave = heat_text # 텍스트 그대로
        except (NoSuchElementException, IndexError) as e:
            print(f"폭염영향 처리 오류: {e}")
        heatwave_impacts.append(cleaned_heatwave)

    # --- 최종 결과 출력 ---
    # (이전과 동일)
    print("\n--- 추출 완료된 리스트 ---")
    print(f"시각: {times}")
    print(f"날씨: {weathers}")
    print(f"기온(℃): {temperatures}")
    print(f"체감기온(℃): {feels_like_temps}")
    print(f"강수량(mm): {precip_amounts}")
    print(f"강수강도: {precip_intensities}")
    print(f"강수확률(%): {precip_probabilities}")
    print(f"바람방향: {wind_directions}")
    print(f"바람속도(m/s): {wind_speeds}")
    print(f"습도(%): {humidities}")
    print(f"폭염영향: {heatwave_impacts}")

except NoSuchElementException as e:
    print(f"오류: 필수 요소를 찾을 수 없습니다. CSS 선택자를 확인하세요. ({e})")
except Exception as e:
    print(f"예상치 못한 오류 발생: {e}")
    import traceback
    traceback.print_exc()

# finally:
#     # 작업 완료 후 드라이버 종료
#     # driver.quit()

총 6개의 시간대 데이터를 찾았습니다.

--- 추출 완료된 리스트 ---
시각: ['19시', '20시', '21시', '22시', '23시', '0시']
날씨: ['맑음', '맑음', '맑음', '맑음', '맑음', '맑음']
기온(℃): [15, 14, 12, 11, 11, 10]
체감기온(℃): [15, 14, 12, 11, 11, 9]
강수량(mm): [None, None, None, None, None, None]
강수강도: [None, None, None, None, None, None]
강수확률(%): [None, None, None, None, None, None]
바람방향: ['남서풍', '서풍', '북서풍', '북서풍', '북서풍', '북서풍']
바람속도(m/s): [2, 1, 2, 3, 3, 3]
습도(%): [40, 40, 45, 45, 45, 50]
폭염영향: [None, None, None, None, None, None]


In [9]:
keys = ['시각', '날씨', '기온(℃)', '체감기온(℃)', '강수량(mm)', '강수강도', '강수확률(%)', '바람방향', '바람속도(m/s)', '습도(%)', '폭염영향']
# 제공된 리스트 변수들을 사용한다고 가정 (times, weathers, temperatures 등)
list_of_lists = [times, weathers, temperatures, feels_like_temps, precip_amounts, precip_intensities, precip_probabilities, wind_directions, wind_speeds, humidities, heatwave_impacts]

structured_data = []
num_items = len(times) # 모든 리스트 길이가 같다고 가정

for i in range(num_items):
    record = {}
    for j, key in enumerate(keys):
         # list_of_lists[j][i] 를 사용하여 올바른 값에 접근
         record[key] = list_of_lists[j][i] 
    structured_data.append(record)

# 이제 structured_data를 JSON으로 변환하여 API에 전달할 수 있습니다.
import json
json_data = json.dumps(structured_data, ensure_ascii=False, indent=2) 
print(type(json_data))
print(json_data)

<class 'str'>
[
  {
    "시각": "19시",
    "날씨": "맑음",
    "기온(℃)": 15,
    "체감기온(℃)": 15,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 40,
    "폭염영향": null
  },
  {
    "시각": "20시",
    "날씨": "맑음",
    "기온(℃)": 14,
    "체감기온(℃)": 14,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 1,
    "습도(%)": 40,
    "폭염영향": null
  },
  {
    "시각": "21시",
    "날씨": "맑음",
    "기온(℃)": 12,
    "체감기온(℃)": 12,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 45,
    "폭염영향": null
  },
  {
    "시각": "22시",
    "날씨": "맑음",
    "기온(℃)": 11,
    "체감기온(℃)": 11,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 3,
    "습도(%)": 45,
    "폭염영향": null
  },
  {
    "시각": "23시",
    "날씨": "맑음",
    "기온(℃)": 11,
    "체감기온(℃)": 11,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방

# GEMINI API 연동

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY 환경 변수를 설정해주세요.")

In [11]:
%pip install -q -U google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.2 -> 25.0.1
[notice] To update, run: c:\Users\dandycode\.pyenv\pyenv-win\versions\3.11.7\python.exe -m pip install --upgrade pip


In [12]:
from google import genai

client = genai.Client(api_key=gemini_api_key)

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Explain how AI works in a few words",
)

print(response.text)

AI learns patterns from data to make predictions or decisions.



In [13]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 날씨 상황을 요약하고, 특히 주목할 만한 변화(예: 강수 시작/종료, 풍속 변화 등)를 설명해주세요.

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 날씨 상황을 요약하고, 특히 주목할 만한 변화(예: 강수 시작/종료, 풍속 변화 등)를 설명해주세요.

**날씨 데이터:**
```json
[
  {
    "시각": "19시",
    "날씨": "맑음",
    "기온(℃)": 15,
    "체감기온(℃)": 15,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 40,
    "폭염영향": null
  },
  {
    "시각": "20시",
    "날씨": "맑음",
    "기온(℃)": 14,
    "체감기온(℃)": 14,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 1,
    "습도(%)": 40,
    "폭염영향": null
  },
  {
    "시각": "21시",
    "날씨": "맑음",
    "기온(℃)": 12,
    "체감기온(℃)": 12,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 45,
    "폭염영향": null
  },
  {
    "시각": "22시",
    "날씨": "맑음",
    "기온(℃)": 11,
    "체감기온(℃)": 11,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 3,
    "습도(%)": 45,
    "폭염영향": null
  },
  {
    "시각": "23시",
    "날씨": "맑음",
   

In [14]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt,
)
print(response.text)

## 날씨 상황 요약

19시부터 0시까지 전반적으로 **맑은 날씨**가 유지되었습니다. 강수량은 기록되지 않았으며, 강수 확률도 없습니다. 기온은 **점차 하강**하는 추세이며, 습도는 약간 증가했습니다.

## 주목할 만한 변화

*   **기온 변화:** 19시에 15℃였던 기온이 0시에는 10℃로 **5℃ 하강**했습니다. 체감 기온 역시 비슷한 수준으로 낮아졌습니다.
*   **바람 변화:**
    *   바람 방향은 19시에는 남서풍에서 20시에 서풍, 그리고 21시부터는 북서풍으로 **변화**했습니다.
    *   바람 속도는 19시에 2m/s, 20시에 1m/s로 감소했다가 21시부터 0시까지 **3m/s로 증가**하여 유지됩니다.
*   **습도 변화:** 습도는 19시와 20시에 40%였으나, 21시부터 약간씩 증가하여 0시에는 50%가 되었습니다. 습도의 증가는 기온 하강과 함께 약간의 쌀쌀함을 느끼게 할 수 있습니다.

**종합:**

시간이 지남에 따라 기온이 낮아지고 바람이 다소 강해지면서 약간 쌀쌀해지는 날씨입니다. 옷차림에 유의해야 하며, 특히 바람의 방향 변화에 따라 체감 온도가 더 낮아질 수 있습니다.


In [15]:
import datetime
current_time_local = datetime.datetime.now()
formatted_time= current_time_local.strftime("%Y-%m-%d %H:%M:%S")
formatted_time

'2025-04-25 18:42:19'

In [16]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 우산이 필요할지 알려주세요.

**오늘 날짜 시간:** {formatted_time}

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 우산이 필요할지 알려주세요.

**오늘 날짜 시간:** 2025-04-25 18:42:19

**날씨 데이터:**
```json
[
  {
    "시각": "19시",
    "날씨": "맑음",
    "기온(℃)": 15,
    "체감기온(℃)": 15,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 40,
    "폭염영향": null
  },
  {
    "시각": "20시",
    "날씨": "맑음",
    "기온(℃)": 14,
    "체감기온(℃)": 14,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 1,
    "습도(%)": 40,
    "폭염영향": null
  },
  {
    "시각": "21시",
    "날씨": "맑음",
    "기온(℃)": 12,
    "체감기온(℃)": 12,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 45,
    "폭염영향": null
  },
  {
    "시각": "22시",
    "날씨": "맑음",
    "기온(℃)": 11,
    "체감기온(℃)": 11,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 3,
    "습도(%)": 45,
    "폭염영향": null
  },
  {
    "시각": "23시",
    "날씨": "맑음"

In [17]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt,
)
print(response.text)

현재 시간은 18시 42분입니다. 가장 가까운 시간대인 19시부터 날씨 데이터를 확인한 결과, 강수량, 강수강도, 강수확률 모두 null 값으로 비가 올 가능성이 없습니다. 따라서 지금 외출할 때 우산은 필요하지 않습니다.



In [18]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요.

**오늘 날짜 시간:** {formatted_time}

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요.

**오늘 날짜 시간:** 2025-04-25 18:42:19

**날씨 데이터:**
```json
[
  {
    "시각": "19시",
    "날씨": "맑음",
    "기온(℃)": 15,
    "체감기온(℃)": 15,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 40,
    "폭염영향": null
  },
  {
    "시각": "20시",
    "날씨": "맑음",
    "기온(℃)": 14,
    "체감기온(℃)": 14,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 1,
    "습도(%)": 40,
    "폭염영향": null
  },
  {
    "시각": "21시",
    "날씨": "맑음",
    "기온(℃)": 12,
    "체감기온(℃)": 12,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 45,
    "폭염영향": null
  },
  {
    "시각": "22시",
    "날씨": "맑음",
    "기온(℃)": 11,
    "체감기온(℃)": 11,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 3,
    "습도(%)": 45,
    "폭염영향": null
  },
  {
    "시각": "23시",
  

In [19]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt,
)
print(response.text)

2025년 4월 25일 18시 42분 현재, 19시부터 0시까지의 날씨 데이터를 분석한 결과 다음과 같은 드레스 코디를 추천합니다.

**날씨 분석:**

*   **날씨:** 맑음 (비 올 확률 없음)
*   **기온:** 현재 15℃에서 점차 낮아져 0시에는 10℃까지 떨어짐
*   **체감온도:** 기온과 거의 비슷
*   **바람:** 2-3 m/s 로 약간 부는 정도
*   **습도:** 40-50%로 건조한 편

**추천 드레스 코디:**

15℃에서 10℃까지 기온이 떨어지는 것을 고려하여, 겉옷을 준비하는 것이 좋습니다.

*   **상의:** 긴팔 티셔츠 또는 얇은 니트
*   **하의:** 청바지, 면바지, 스커트 등 편안한 스타일
*   **겉옷:**
    *   얇은 가디건 또는 바람막이 점퍼: 초저녁 (19-21시) 에 적합
    *   얇은 트렌치 코트 또는 야상 점퍼: 늦은 밤 (22시 이후) 에 적합
*   **신발:** 운동화, 단화 등 편안한 신발
*   **액세서리:** 스카프 또는 머플러 (선택 사항, 저녁에 기온이 더 떨어질 경우 대비)

**추가 팁:**

*   개인의 추위를 타는 정도에 따라 옷을 조절하세요.
*   활동량에 따라 옷의 두께를 조절하세요.
*   갑작스러운 기온 변화에 대비하여 여벌의 옷을 준비하는 것도 좋습니다.

이 코디는 일반적인 추천이며, 개인의 취향과 상황에 따라 자유롭게 변형하여 입으시면 됩니다.



In [25]:
formatted_time= current_time_local.strftime("%Y%m%d")

file = open(f"result_{formatted_time}.md", "w", encoding="utf8")

file.write(str(response.text))
file.close()

In [46]:
prompt = f"""다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요. 결과는 현재 날씨에 어울리는 CSS 스타일을 적용한 HTML로 작성해주세요. HTML 문서 외 설명은 작성하지 마세요.

**오늘 날짜 시간:** {formatted_time}

**날씨 데이터:**
```json
{json_data}
```
"""

print(prompt)

다음은 시간대별 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요. 결과는 현재 날씨에 어울리는 CSS 스타일을 적용한 HTML로 작성해주세요. HTML 문서 외 설명은 작성하지 마세요.

**오늘 날짜 시간:** 20250425

**날씨 데이터:**
```json
[
  {
    "시각": "19시",
    "날씨": "맑음",
    "기온(℃)": 15,
    "체감기온(℃)": 15,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "남서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 40,
    "폭염영향": null
  },
  {
    "시각": "20시",
    "날씨": "맑음",
    "기온(℃)": 14,
    "체감기온(℃)": 14,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "서풍",
    "바람속도(m/s)": 1,
    "습도(%)": 40,
    "폭염영향": null
  },
  {
    "시각": "21시",
    "날씨": "맑음",
    "기온(℃)": 12,
    "체감기온(℃)": 12,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 2,
    "습도(%)": 45,
    "폭염영향": null
  },
  {
    "시각": "22시",
    "날씨": "맑음",
    "기온(℃)": 11,
    "체감기온(℃)": 11,
    "강수량(mm)": null,
    "강수강도": null,
    "강수확률(%)": null,
    "바람방향": "북서풍",
    "바람속도(m/s)": 3,
    "습

In [47]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt,
)
print(response.text)

```html
<!DOCTYPE html>
<html>
<head>
<title>오늘의 추천 코디</title>
<style>
body {
    font-family: sans-serif;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    min-height: 100vh;
    margin: 0;
    background-color: #f0f8ff; /* Light sky blue background */
}

.container {
    background-color: #ffffff;
    border-radius: 10px;
    padding: 20px;
    box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);
    text-align: center;
    width: 80%;
    max-width: 600px;
}

h1 {
    color: #333;
    margin-bottom: 20px;
}

.suggestion {
    font-size: 18px;
    color: #555;
    margin-bottom: 30px;
}

.clothing-items {
    display: flex;
    justify-content: space-around;
    flex-wrap: wrap;
}

.clothing-item {
    background-color: #e6f7ff; /* Lighter sky blue */
    border: 1px solid #add8e6; /* Light blue border */
    border-radius: 8px;
    padding: 15px;
    margin: 10px;
    width: calc(50% - 20px);
    box-sizing: border-box;
}

.clothing-item h

In [48]:
result = response.text
result = result.replace("```html","").replace("```","")
print(result)


<!DOCTYPE html>
<html>
<head>
<title>오늘의 추천 코디</title>
<style>
body {
    font-family: sans-serif;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    min-height: 100vh;
    margin: 0;
    background-color: #f0f8ff; /* Light sky blue background */
}

.container {
    background-color: #ffffff;
    border-radius: 10px;
    padding: 20px;
    box-shadow: 0 4px 8px rgba(0, 0, 0, 0.1);
    text-align: center;
    width: 80%;
    max-width: 600px;
}

h1 {
    color: #333;
    margin-bottom: 20px;
}

.suggestion {
    font-size: 18px;
    color: #555;
    margin-bottom: 30px;
}

.clothing-items {
    display: flex;
    justify-content: space-around;
    flex-wrap: wrap;
}

.clothing-item {
    background-color: #e6f7ff; /* Lighter sky blue */
    border: 1px solid #add8e6; /* Light blue border */
    border-radius: 8px;
    padding: 15px;
    margin: 10px;
    width: calc(50% - 20px);
    box-sizing: border-box;
}

.clothing-item h3 {
   

In [49]:
formatted_time= current_time_local.strftime("%Y%m%d")

file = open(f"result_{formatted_time}.html", "w", encoding="utf8")

file.write(str(result))
file.close()